In [4]:

# ============================================================
# MODEL EVALUATION, OPTIMIZATION AND SELECTION
# ============================================================

import os
import json
import glob
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

cwd = os.getcwd()

PROJECT_ROOT = (
    os.path.dirname(cwd)
    if os.path.basename(cwd).lower() == "notebooks"
    else cwd
)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load forecasting dataset
# ------------------------------------------------------------

dataset_path = os.path.join(
    OUTPUT_DIR,
    "forecasting_framework_dataset.csv"
)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        "forecasting_framework_dataset.csv not found."
    )

df = pd.read_csv(dataset_path)

df.columns = (
    df.columns.astype(str).str.strip().str.lower()
)

date_candidates = [
    "date",
    "report_date",
    "observation_date",
    "snapshot_date"
]

date_col = next(
    (c for c in date_candidates if c in df.columns),
    None
)

if date_col is None:
    raise ValueError("Date column not found.")

target_col = None

config_path = os.path.join(
    OUTPUT_DIR,
    "machine_learning_configuration.json"
)

if os.path.exists(config_path):

    with open(config_path, "r") as file:
        ml_config = json.load(file)

    configured_target = ml_config.get("target_variable")

    if configured_target in df.columns:
        target_col = configured_target

if target_col is None:

    config_path = os.path.join(
        OUTPUT_DIR,
        "statistical_forecasting_configuration.json"
    )

    if os.path.exists(config_path):

        with open(config_path, "r") as file:
            statistical_config = json.load(file)

        configured_target = statistical_config.get(
            "target_variable"
        )

        if configured_target in df.columns:
            target_col = configured_target

if target_col is None:

    target_candidates = [
        "children_in_hhs_care",
        "total_children",
        "total_care_load",
        "care_load",
        "placement_demand"
    ]

    target_col = next(
        (c for c in target_candidates if c in df.columns),
        None
    )

if target_col is None:

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    numeric_columns = [
        c for c in numeric_columns
        if not any(
            term in c
            for term in ["lag_", "rolling_", "moving_average"]
        )
    ]

    if len(numeric_columns) == 1:
        target_col = numeric_columns[0]
    else:
        raise ValueError(
            "Unable to identify target variable safely."
        )

df[date_col] = pd.to_datetime(
    df[date_col],
    errors="coerce"
)

df[target_col] = pd.to_numeric(
    df[target_col],
    errors="coerce"
)

df = (
    df.dropna(subset=[date_col, target_col])
    .sort_values(date_col)
    .drop_duplicates(subset=[date_col])
    .reset_index(drop=True)
)

print("Dataset loaded.")
print("Target:", target_col)
print("Date column:", date_col)
print("Observations:", len(df))

# ------------------------------------------------------------
# Load chronological boundaries
# ------------------------------------------------------------

split_path = os.path.join(
    OUTPUT_DIR,
    "chronological_split_summary.csv"
)

if not os.path.exists(split_path):
    raise FileNotFoundError(
        "chronological_split_summary.csv not found."
    )

split_summary = pd.read_csv(split_path)

def get_split_date(dataset_name):

    result = split_summary.loc[
        split_summary["Dataset"].astype(str).str.lower()
        == dataset_name.lower(),
        "End_Date"
    ]

    if result.empty:
        raise ValueError(
            f"{dataset_name} end date not found."
        )

    return pd.to_datetime(result.iloc[0])


train_end = get_split_date("Training")
validation_end = get_split_date("Validation")

# ------------------------------------------------------------
# Identify prediction columns
# ------------------------------------------------------------

def find_column(data, candidates):

    normalized = {
        str(column).strip().lower(): column
        for column in data.columns
    }

    for candidate in candidates:

        if candidate.lower() in normalized:
            return normalized[candidate.lower()]

    return None


def normalize_horizon(value):

    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    if value in ["1", "1_day", "1-day", "day_1"]:
        return "1_day"

    if value in ["7", "7_day", "7-day", "day_7"]:
        return "7_day"

    if value in ["14", "14_day", "14-day", "day_14"]:
        return "14_day"

    if value in ["30", "30_day", "30-day", "day_30"]:
        return "30_day"

    if "1_day" in value or "1-day" in value:
        return "1_day"

    if "7_day" in value or "7-day" in value:
        return "7_day"

    if "14_day" in value or "14-day" in value:
        return "14_day"

    if "30_day" in value or "30-day" in value:
        return "30_day"

    return None


def normalize_prediction_file(file_path):

    data = pd.read_csv(file_path)

    if data.empty:
        return pd.DataFrame()

    data.columns = (
        data.columns.astype(str).str.strip()
    )

    actual_col = find_column(
        data,
        ["Actual", "Actual_Value", "y_true", "Observed"]
    )

    predicted_col = find_column(
        data,
        ["Predicted", "Prediction", "Forecast", "y_pred"]
    )

    if actual_col is None or predicted_col is None:
        return pd.DataFrame()

    model_col = find_column(
        data,
        ["Model", "Model_Name", "Algorithm"]
    )

    horizon_col = find_column(
        data,
        ["Horizon", "Forecast_Horizon", "Horizon_Name"]
    )

    target_date_col = find_column(
        data,
        [
            "Actual_Target_Date",
            "Target_Date",
            "Forecast_Date",
            "Date"
        ]
    )

    origin_col = find_column(
        data,
        ["Forecast_Origin", "Origin_Date"]
    )

    partition_col = find_column(
        data,
        ["Dataset", "Partition", "Split", "Data_Type"]
    )

    method_col = find_column(
        data,
        ["Forecasting_Method", "Method"]
    )

    result = pd.DataFrame()

    result["Model"] = (
        data[model_col].astype(str)
        if model_col else
        os.path.basename(file_path)
    )

    result["Actual"] = pd.to_numeric(
        data[actual_col],
        errors="coerce"
    )

    result["Predicted"] = pd.to_numeric(
        data[predicted_col],
        errors="coerce"
    )

    if target_date_col:

        result["Actual_Target_Date"] = pd.to_datetime(
            data[target_date_col],
            errors="coerce"
        )

    else:

        result["Actual_Target_Date"] = pd.NaT

    if origin_col:

        result["Forecast_Origin"] = pd.to_datetime(
            data[origin_col],
            errors="coerce"
        )

    else:

        result["Forecast_Origin"] = pd.NaT

    if horizon_col:

        result["Horizon"] = data[horizon_col].apply(
            normalize_horizon
        )

    else:

        result["Horizon"] = None

    if partition_col:

        result["Partition"] = (
            data[partition_col].astype(str).str.lower()
        )

    else:

        result["Partition"] = ""

    if method_col:

        result["Forecasting_Method"] = (
            data[method_col].astype(str)
        )

    else:

        result["Forecasting_Method"] = "Standard"

    # Identify model family from model name and source file.

    def identify_family(model_name, filename):

        combined = (
            str(model_name) + " " + str(filename)
        ).lower()

        if any(
            term in combined
            for term in [
                "random_forest",
                "gradient_boosting",
                "recursive",
                "machine_learning"
            ]
        ):
            return "Machine Learning"

        if any(
            term in combined
            for term in [
                "arima",
                "sarima",
                "holt",
                "exponential",
                "statistical"
            ]
        ):
            return "Statistical"

        if any(
            term in combined
            for term in [
                "naive",
                "moving_average",
                "baseline",
                "persistence",
                "seasonal"
            ]
        ):
            return "Baseline"

        return "Other"

    result["Model_Family"] = result["Model"].apply(
        lambda x: identify_family(
            x,
            os.path.basename(file_path)
        )
    )

    # Identify partition when not explicitly supplied.

    result["Partition"] = result["Partition"].replace(
        {
            "validation": "validation",
            "valid": "validation",
            "testing": "testing",
            "test": "testing"
        }
    )

    missing_partition = result["Partition"].isin(
        ["", "nan", "none"]
    )

    result.loc[
        missing_partition
        & (result["Actual_Target_Date"] <= validation_end),
        "Partition"
    ] = "validation"

    result.loc[
        missing_partition
        & (result["Actual_Target_Date"] > validation_end),
        "Partition"
    ] = "testing"

    # Infer horizon from target date and origin when possible.

    missing_horizon = result["Horizon"].isna()

    inferred_steps = (
        result["Actual_Target_Date"]
        - result["Forecast_Origin"]
    ).dt.days

    if missing_horizon.any():

        for steps, horizon_name in [
            (1, "1_day"),
            (7, "7_day"),
            (14, "14_day"),
            (30, "30_day")
        ]:

            mask = (
                missing_horizon
                & (inferred_steps == steps)
            )

            result.loc[mask, "Horizon"] = horizon_name

    # Infer horizon from forecast file metadata when available.

    filename = os.path.basename(file_path).lower()

    for horizon_name in [
        "1_day",
        "7_day",
        "14_day",
        "30_day"
    ]:

        if horizon_name in filename:

            result.loc[
                result["Horizon"].isna(),
                "Horizon"
            ] = horizon_name

    return result


# ------------------------------------------------------------
# Find all prediction files
# ------------------------------------------------------------

prediction_files = glob.glob(
    os.path.join(OUTPUT_DIR, "*predictions*.csv")
)

prediction_files += glob.glob(
    os.path.join(OUTPUT_DIR, "*forecast*.csv")
)

prediction_files = sorted(set(prediction_files))

prediction_frames = []

excluded_files = []

for file_path in prediction_files:

    filename = os.path.basename(file_path).lower()

    if any(
        term in filename
        for term in [
            "error",
            "residual",
            "configuration"
        ]
    ):
        continue

    try:

        normalized = normalize_prediction_file(
            file_path
        )

        if not normalized.empty:

            prediction_frames.append(normalized)

            print(
                "Loaded:",
                os.path.basename(file_path),
                "| Records:",
                len(normalized)
            )

        else:

            excluded_files.append(
                os.path.basename(file_path)
            )

    except Exception as error:

        print(
            "Skipped:",
            os.path.basename(file_path),
            "| Reason:",
            error
        )

if not prediction_frames:

    raise FileNotFoundError(
        "No usable prediction files found in outputs. "
        "Run the baseline, statistical and machine learning "
        "forecasting notebooks first."
    )

all_predictions = pd.concat(
    prediction_frames,
    ignore_index=True
)

# ------------------------------------------------------------
# Clean and validate combined predictions
# ------------------------------------------------------------

all_predictions["Actual"] = pd.to_numeric(
    all_predictions["Actual"],
    errors="coerce"
)

all_predictions["Predicted"] = pd.to_numeric(
    all_predictions["Predicted"],
    errors="coerce"
)

all_predictions = all_predictions.replace(
    [np.inf, -np.inf],
    np.nan
)

all_predictions = all_predictions.dropna(
    subset=["Actual", "Predicted", "Horizon"]
).copy()

all_predictions = all_predictions[
    all_predictions["Horizon"].isin(
        ["1_day", "7_day", "14_day", "30_day"]
    )
].copy()

all_predictions = all_predictions[
    all_predictions["Partition"].isin(
        ["validation", "testing"]
    )
].copy()

if all_predictions.empty:
    raise ValueError(
        "No predictions remain after cleaning. "
        "Check horizon and partition columns in prediction files."
    )

all_predictions["Residual"] = (
    all_predictions["Actual"]
    - all_predictions["Predicted"]
)

all_predictions["Absolute_Error"] = (
    all_predictions["Residual"].abs()
)

all_predictions["Squared_Error"] = (
    all_predictions["Residual"] ** 2
)

all_predictions["Absolute_Actual"] = (
    all_predictions["Actual"].abs()
)

all_predictions["Absolute_Percentage_Error"] = np.where(
    all_predictions["Actual"] != 0,
    all_predictions["Absolute_Error"]
    / all_predictions["Actual"].abs() * 100,
    np.nan
)

print("\nCombined prediction records:", len(all_predictions))
print("\nModel families:")
print(all_predictions["Model_Family"].value_counts())

# ------------------------------------------------------------
# Calculate comprehensive evaluation metrics
# ------------------------------------------------------------

def calculate_metrics(group):

    actual = group["Actual"].values
    predicted = group["Predicted"].values

    residuals = actual - predicted

    absolute_actual_sum = np.sum(np.abs(actual))

    wape = (
        np.sum(np.abs(residuals))
        / absolute_actual_sum * 100
        if absolute_actual_sum != 0
        else np.nan
    )

    nonzero = actual != 0

    mape = (
        mean_absolute_percentage_error(
            actual[nonzero],
            predicted[nonzero]
        ) * 100
        if nonzero.any()
        else np.nan
    )

    return {

        "Observations": len(group),

        "MAE": mean_absolute_error(
            actual,
            predicted
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        ),

        "MAPE_Percent": mape,

        "WAPE_Percent": wape,

        "Forecast_Bias": np.mean(
            predicted - actual
        ),

        "Mean_Absolute_Residual": np.mean(
            np.abs(residuals)
        ),

        "Residual_Mean": np.mean(residuals),

        "Residual_Std": (
            np.std(residuals, ddof=1)
            if len(residuals) > 1
            else np.nan
        ),

        "Residual_Min": np.min(residuals),

        "Residual_Max": np.max(residuals),

        "Residual_Median": np.median(residuals)

    }


# ------------------------------------------------------------
# Calculate model-level performance comparison
# ------------------------------------------------------------

comparison_results = []

for (partition, family, model, horizon), group in (
    all_predictions.groupby(
        [
            "Partition",
            "Model_Family",
            "Model",
            "Horizon"
        ]
    )
):

    metrics = calculate_metrics(group)

    comparison_results.append({

        "Partition": partition,
        "Model_Family": family,
        "Model": model,
        "Horizon": horizon,
        **metrics

    })

model_performance = pd.DataFrame(
    comparison_results
)

# ------------------------------------------------------------
# Calculate horizon-wise performance
# ------------------------------------------------------------

horizon_results = []

for (partition, horizon), group in (
    all_predictions.groupby(
        ["Partition", "Horizon"]
    )
):

    horizon_results.append({

        "Partition": partition,
        "Horizon": horizon,
        "Model_Count": group["Model"].nunique(),
        **calculate_metrics(group)

    })

horizon_wise_metrics = pd.DataFrame(
    horizon_results
)

# ------------------------------------------------------------
# Compare all model families
# ------------------------------------------------------------

family_results = []

for (partition, family, horizon), group in (
    all_predictions.groupby(
        [
            "Partition",
            "Model_Family",
            "Horizon"
        ]
    )
):

    family_results.append({

        "Partition": partition,
        "Model_Family": family,
        "Horizon": horizon,
        **calculate_metrics(group)

    })

family_comparison = pd.DataFrame(
    family_results
)

# ------------------------------------------------------------
# Evaluate surge and stable periods
# ------------------------------------------------------------

training_values = df.loc[
    df[date_col] <= train_end,
    target_col
].astype(float)

rolling_values = training_values.rolling(
    window=7,
    min_periods=3
).mean().dropna()

if rolling_values.empty:

    surge_threshold = training_values.quantile(0.90)

else:

    surge_threshold = rolling_values.quantile(0.90)

actual_periods = df[
    [date_col, target_col]
].copy()

actual_periods = actual_periods.rename(
    columns={
        date_col: "Actual_Target_Date",
        target_col: "Observed_Target"
    }
)

actual_periods["Rolling_7_Observation_Mean"] = (
    actual_periods["Observed_Target"]
    .rolling(7, min_periods=3)
    .mean()
)

actual_periods["Period_Type"] = np.where(
    actual_periods["Rolling_7_Observation_Mean"]
    > surge_threshold,
    "Surge",
    "Stable"
)

all_predictions = all_predictions.merge(
    actual_periods[
        [
            "Actual_Target_Date",
            "Period_Type",
            "Rolling_7_Observation_Mean"
        ]
    ],
    on="Actual_Target_Date",
    how="left"
)

all_predictions["Period_Type"] = (
    all_predictions["Period_Type"].fillna("Unclassified")
)

# ------------------------------------------------------------
# Calculate surge and stable period performance
# ------------------------------------------------------------

period_results = []

classified_predictions = all_predictions[
    all_predictions["Period_Type"].isin(
        ["Surge", "Stable"]
    )
]

for (
    partition,
    family,
    model,
    horizon,
    period_type
), group in classified_predictions.groupby(
    [
        "Partition",
        "Model_Family",
        "Model",
        "Horizon",
        "Period_Type"
    ]
):

    period_results.append({

        "Partition": partition,
        "Model_Family": family,
        "Model": model,
        "Horizon": horizon,
        "Period_Type": period_type,
        **calculate_metrics(group)

    })

surge_stable_metrics = pd.DataFrame(
    period_results
)

# ------------------------------------------------------------
# Analyze hyperparameter sensitivity
# ------------------------------------------------------------

tuning_path = os.path.join(
    OUTPUT_DIR,
    "machine_learning_hyperparameter_results.csv"
)

if os.path.exists(tuning_path):

    tuning_results = pd.read_csv(tuning_path)

    if "Best_CV_MAE" in tuning_results.columns:

        tuning_results["Best_CV_MAE"] = pd.to_numeric(
            tuning_results["Best_CV_MAE"],
            errors="coerce"
        )

        tuning_results["Sensitivity_Rank"] = (
            tuning_results.groupby("Model")[
                "Best_CV_MAE"
            ].rank(method="min")
        )

        tuning_results["Performance_Deviation"] = (
            tuning_results.groupby("Model")[
                "Best_CV_MAE"
            ].transform("max")
            -
            tuning_results.groupby("Model")[
                "Best_CV_MAE"
            ].transform("min")
        )

    else:

        tuning_results["Sensitivity_Rank"] = np.nan
        tuning_results["Performance_Deviation"] = np.nan

else:

    tuning_results = pd.DataFrame(
        columns=[
            "Model",
            "Horizon",
            "Best_CV_MAE",
            "Best_Parameters",
            "Sensitivity_Rank",
            "Performance_Deviation"
        ]
    )

# ------------------------------------------------------------
# Compare forecast accuracy and stability
# ------------------------------------------------------------

validation_performance = model_performance[
    model_performance["Partition"] == "validation"
].copy()

if validation_performance.empty:
    raise ValueError(
        "No validation metrics available for model selection."
    )

validation_performance["Accuracy_Rank"] = (
    validation_performance.groupby("Horizon")["MAE"]
    .rank(method="min", ascending=True)
)

validation_performance["RMSE_Rank"] = (
    validation_performance.groupby("Horizon")["RMSE"]
    .rank(method="min", ascending=True)
)

validation_performance["WAPE_Rank"] = (
    validation_performance.groupby("Horizon")["WAPE_Percent"]
    .rank(method="min", ascending=True)
)

validation_performance["Bias_Absolute"] = (
    validation_performance["Forecast_Bias"].abs()
)

validation_performance["Bias_Rank"] = (
    validation_performance.groupby("Horizon")[
        "Bias_Absolute"
    ].rank(method="min", ascending=True)
)

validation_performance["Selection_Score"] = (
    validation_performance["Accuracy_Rank"] * 0.50
    + validation_performance["RMSE_Rank"] * 0.20
    + validation_performance["WAPE_Rank"] * 0.20
    + validation_performance["Bias_Rank"] * 0.10
)

# ------------------------------------------------------------
# Select best model for each forecast horizon
# ------------------------------------------------------------

best_model_summary = (
    validation_performance
    .sort_values(
        ["Horizon", "Selection_Score", "MAE"]
    )
    .groupby("Horizon")
    .first()
    .reset_index()
)

best_model_summary["Selection_Basis"] = (
    "Validation performance: MAE 50%, RMSE 20%, "
    "WAPE 20%, absolute bias 10%"
)

# ------------------------------------------------------------
# Attach held-out test performance to selected models
# ------------------------------------------------------------

test_performance = model_performance[
    model_performance["Partition"] == "testing"
].copy()

if not test_performance.empty:

    best_model_summary = best_model_summary.merge(
        test_performance[
            [
                "Model",
                "Horizon",
                "MAE",
                "RMSE",
                "MAPE_Percent",
                "WAPE_Percent",
                "Forecast_Bias",
                "Observations"
            ]
        ].rename(
            columns={
                "MAE": "Test_MAE",
                "RMSE": "Test_RMSE",
                "MAPE_Percent": "Test_MAPE_Percent",
                "WAPE_Percent": "Test_WAPE_Percent",
                "Forecast_Bias": "Test_Forecast_Bias",
                "Observations": "Test_Observations"
            }
        ),
        on=["Model", "Horizon"],
        how="left"
    )

# ------------------------------------------------------------
# Establish final model selection criteria
# ------------------------------------------------------------

selection_criteria = {

    "Primary_Metric": "MAE",
    "Secondary_Metric": "RMSE",
    "Relative_Error_Metric": "WAPE",
    "Bias_Requirement": "Prefer lower absolute forecast bias",
    "Selection_Data": "Validation predictions only",
    "Final_Reporting_Data": "Held-out testing predictions",
    "Horizon_Selection": "Independent selection for each horizon",
    "Selection_Weights": {
        "MAE_Rank": 0.50,
        "RMSE_Rank": 0.20,
        "WAPE_Rank": 0.20,
        "Absolute_Bias_Rank": 0.10
    },
    "Surge_Definition": (
        "Seven-observation rolling mean above the "
        "training-period 90th percentile"
    ),
    "MAPE_Handling": (
        "Calculated only for nonzero actual values"
    )

}

# ------------------------------------------------------------
# Save combined prediction dataset
# ------------------------------------------------------------

all_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "combined_model_predictions.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save required model comparison deliverable
# ------------------------------------------------------------

model_performance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "model_performance_comparison.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save required horizon metrics deliverable
# ------------------------------------------------------------

horizon_wise_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "horizon_wise_metrics.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save required best model deliverable
# ------------------------------------------------------------

best_model_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "best_model_summary.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save additional evaluation outputs
# ------------------------------------------------------------

family_comparison.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "model_family_comparison.csv"
    ),
    index=False
)

surge_stable_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "surge_stable_performance.csv"
    ),
    index=False
)

tuning_results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "hyperparameter_sensitivity_analysis.csv"
    ),
    index=False
)

validation_performance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "validation_model_ranking.csv"
    ),
    index=False
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "model_selection_criteria.json"
    ),
    "w"
) as file:

    json.dump(
        selection_criteria,
        file,
        indent=4
    )

# ------------------------------------------------------------
# Final evaluation summary
# ------------------------------------------------------------

print("\n==============================================")
print("MODEL EVALUATION COMPLETED")
print("==============================================")

print("\nTarget variable:", target_col)

print("\nCombined prediction records:", len(all_predictions))

print("\nModel families:")
print(
    all_predictions["Model_Family"]
    .value_counts()
    .to_string()
)

print("\nHORIZON-WISE METRICS")
print(
    horizon_wise_metrics.to_string(index=False)
)

print("\nMODEL FAMILY COMPARISON")
print(
    family_comparison.to_string(index=False)
)

print("\nSURGE AND STABLE PERIOD PERFORMANCE")
print(
    surge_stable_metrics.to_string(index=False)
)

print("\nHYPERPARAMETER SENSITIVITY")
print(
    tuning_results.to_string(index=False)
)

print("\nBEST MODEL PER HORIZON")
print(
    best_model_summary.to_string(index=False)
)

print("\nREQUIRED DELIVERABLES")

for filename in [
    "model_performance_comparison.csv",
    "horizon_wise_metrics.csv",
    "best_model_summary.csv"
]:

    print(
        os.path.join(OUTPUT_DIR, filename)
    )

print("\nAll model evaluation outputs saved successfully.")


Dataset loaded.
Target: children_in_hhs_care
Date column: date
Observations: 690
Loaded: baseline_predictions.csv | Records: 3900
Loaded: combined_model_predictions.csv | Records: 6460
Loaded: machine_learning_recursive_predictions.csv | Records: 416
Loaded: machine_learning_test_predictions.csv | Records: 1040
Loaded: machine_learning_validation_predictions.csv | Records: 1040
Loaded: statistical_test_forecasts.csv | Records: 20
Loaded: statistical_validation_forecasts.csv | Records: 20
Loaded: validation_predictions.csv | Records: 416
Loaded: walk_forward_predictions.csv | Records: 416

Combined prediction records: 12920

Model families:
Model_Family
Baseline            7800
Machine Learning    3376
Other               1664
Statistical           80
Name: count, dtype: int64

MODEL EVALUATION COMPLETED

Target variable: children_in_hhs_care

Combined prediction records: 12920

Model families:
Model_Family
Baseline            7800
Machine Learning    3376
Other               1664
Stati